In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
pd.set_option("display.max_columns", 50)

In [3]:
ROOT = Path.cwd().parent
RAW_PATH = ROOT / "data" / "raw" / "telco_churn.csv"

print(RAW_PATH, "| exists:", RAW_PATH.exists())

d:\Pratik\Machine_Learning\Enterprise Customer Intelligence & Churn Prediction Platform\data\raw\telco_churn.csv | exists: True


In [4]:
df = pd.read_csv(RAW_PATH, dtype=str)
df.head(10)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.3,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.7,151.65,Yes
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
6,1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.1,1949.4,No
7,6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.9,No
8,7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.8,3046.05,Yes
9,6388-TABGU,Male,0,No,Yes,62,Yes,No,DSL,Yes,Yes,No,No,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,No


In [5]:
print("Rows, columns:", df.shape)

Rows, columns: (7043, 21)


In [6]:
print(list(df.columns))

['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [7]:
print("Missing (NaN) values per column:")
print(df.isna().sum()[df.isna().sum() > 0])

Missing (NaN) values per column:
Series([], dtype: int64)


In [9]:
# Find hidden blanks
blank_counts = df.apply(lambda s: s.str.strip().eq("").sum())
print("Cells that are empty or only whitespace:")
print(blank_counts[blank_counts > 0])

Cells that are empty or only whitespace:
TotalCharges    11
dtype: int64


In [10]:
blank_total = df[df["TotalCharges"].str.strip() == ""]
blank_total[["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,,No
753,3115-CZMZD,0,20.25,,No
936,5709-LVOEQ,0,80.85,,No
1082,4367-NUYAO,0,25.75,,No
1340,1371-DWPAZ,0,56.05,,No
3331,7644-OMVMY,0,19.85,,No
3826,3213-VVOLG,0,25.35,,No
4380,2520-SGTTA,0,20,,No
5218,2923-ARZLG,0,19.7,,No
6670,4075-WKNIU,0,73.35,,No


In [11]:
# Duplicates
print("Exact duplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df["customerID"].duplicated().sum())
print("Customer ID format OK:", df["customerID"].str.match(r"^\d{4}-[A-Z]{5}$").mean() * 100, "%")

Exact duplicate rows: 0
Duplicate customer IDs: 0
Customer ID format OK: 100.0 %


In [13]:
# Numeric columns, converted safely
num = df[["tenure", "MonthlyCharges", "TotalCharges"]].apply(pd.to_numeric, errors="coerce")
print("Values that are not valid numbers:")
print(num.isna().sum())

Values that are not valid numbers:
tenure             0
MonthlyCharges     0
TotalCharges      11
dtype: int64


In [ ]:
num.describe().round(2)

,tenure,MonthlyCharges,TotalCharges
count,7043.00,7043.00,7032.00
mean,32.37,64.76,2283.30
std,24.56,30.09,2266.77
min,0.00,18.25,18.80
25%,9.00,35.50,401.45
50%,29.00,70.35,1397.48
75%,55.00,89.85,3794.74
max,72.00,118.75,8684.80


In [15]:
# Categorical columns
cat_cols = [c for c in df.columns if c not in ["customerID", "tenure", "MonthlyCharges", "TotalCharges"]]

for col in cat_cols:
    print(f"{col}: {sorted(df[col].dropna().unique())}")

gender: ['Female', 'Male']
SeniorCitizen: ['0', '1']
Partner: ['No', 'Yes']
Dependents: ['No', 'Yes']
PhoneService: ['No', 'Yes']
MultipleLines: ['No', 'No phone service', 'Yes']
InternetService: ['DSL', 'Fiber optic', 'No']
OnlineSecurity: ['No', 'No internet service', 'Yes']
OnlineBackup: ['No', 'No internet service', 'Yes']
DeviceProtection: ['No', 'No internet service', 'Yes']
TechSupport: ['No', 'No internet service', 'Yes']
StreamingTV: ['No', 'No internet service', 'Yes']
StreamingMovies: ['No', 'No internet service', 'Yes']
Contract: ['Month-to-month', 'One year', 'Two year']
PaperlessBilling: ['No', 'Yes']
PaymentMethod: ['Bank transfer (automatic)', 'Credit card (automatic)', 'Electronic check', 'Mailed check']
Churn: ['No', 'Yes']


In [16]:
# Target value
churn_counts = df["Churn"].value_counts()
print(churn_counts)

Churn
No     5174
Yes    1869
Name: count, dtype: int64


In [17]:
print((churn_counts / len(df) * 100).round(1).astype(str) + " %")

Churn
No     73.5 %
Yes    26.5 %
Name: count, dtype: str


In [18]:
# Consistency check
no_internet = df[df["InternetService"] == "No"]
print("Customers with no internet:", len(no_internet))
print(no_internet["OnlineSecurity"].value_counts())

Customers with no internet: 1526
OnlineSecurity
No internet service    1526
Name: count, dtype: int64


In [19]:
no_phone = df[df["PhoneService"] == "No"]
print("\nCustomers with no phone:", len(no_phone))
print(no_phone["MultipleLines"].value_counts())


Customers with no phone: 682
MultipleLines
No phone service    682
Name: count, dtype: int64


In [20]:
# Starter data dictionary
dictionary = pd.DataFrame({
    "column": df.columns,
    "example_value": df.iloc[0].values,
    "unique_values": [df[c].nunique() for c in df.columns],
    "missing_or_blank": [(df[c].isna() | (df[c].str.strip() == "")).sum() for c in df.columns],
})

dictionary

,column,example_value,unique_values,missing_or_blank
0,customerID,7590-VHVEG,7043,0
1,gender,Female,2,0
2,SeniorCitizen,0,2,0
3,Partner,Yes,2,0
4,Dependents,No,2,0
5,tenure,1,73,0
6,PhoneService,No,2,0
7,MultipleLines,No phone service,3,0
8,InternetService,DSL,3,0
9,OnlineSecurity,No,3,0
